# Gold · chamados_nota

**Origem:** silver de `chamados`, `pesquisa`, `motivos` e `chamados_hora` → **Destino:** `vcredit_gold.chamados_nota`

**Grão:** uma linha por chamado. **Depende de:** todas as silvers acima

Base para as análises de satisfação: chamado, nota, motivo e tempos reais de espera e atendimento.

| Regra | Como |
|---|---|
| Nota | Uma por chamado (pesquisas repetidas têm a mesma nota); vazia se o cliente não respondeu |
| Atendimento real (`atend_seg`) | Maior valor entre o registrado e fim − início |
| Espera real (`espera_seg`) | Maior valor entre o registrado e início − abertura; só atendidos em canais humanos |
| Tempo até desistir (`espera_ate_desistir_seg`) | Fim − abertura; só abandonos |
| Faixas de tempo | `faixa_espera_min` e `faixa_atend_min`, de 5 em 5 minutos, até 60 |
| Mês | A partir da hora de abertura |
| Integridade | `assert` garante uma linha por chamado, sem perdas nem duplicações |

**Decisões:** para tempo, use `espera_seg` e `atend_seg`, e não `tempo_espera` e `tempo_atendimento`: o registro original não passa de ~16 min e subestima os atendimentos longos. Abandonos ficam fora de `espera_seg` (mesma regra da `fato_chamados`).

Pipeline idempotente (lê da silver e sobrescreve).

In [0]:
from pyspark.sql import functions as F

s = "rocket_lab.vcredit_silver"
g = "rocket_lab.vcredit_gold"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {g}")
print("schema pronto:", g)

In [0]:
LARGURA, TETO = 5, 60   # minutos (ajuste depois da célula de percentis)

# Tabela de chamados
ch = (spark.table(f"{s}.chamados")
      .select("id_chamado", "motivo", "canal", "resolvido", "tempo_espera", "tempo_atendimento"))

# Tabela de chamados_hora
hr = (spark.table(f"{s}.chamados_hora")
      .select("id_chamado",
              F.col("hora_abertura_chamado").alias("abertura"),
              F.col("hora_inicio_atendimento").alias("inicio"),
              F.col("hora_finalizacao_atendimento").alias("fim"),
              F.col("sem_inicio_atendimento").alias("abandono")))

# Tabela de motivos
mo = (spark.table(f"{s}.motivos")
      .select(F.col("nome_motivo").alias("motivo"), "categoria", "criticidade"))

# Tabela de pesquisa: Uma linha por chamado (as respostas repetidas têm a mesma nota)
pe = (spark.table(f"{s}.pesquisa").groupBy("id_chamado")
        .agg(F.max("nota_atendimento").alias("nota"),
             F.min("nota_atendimento").alias("nota_min"),
             F.count("*").alias("qtd_respostas")))
print("Chamados com mais de uma resposta:", pe.filter("qtd_respostas > 1").count(),
      "| Com notas diferentes entre as respostas:", pe.filter("nota != nota_min").count())
pe = pe.select("id_chamado", "nota", "qtd_respostas")

humano = F.col("canal").isin("atendimento inicial", "atendimento especializado")
espera_atendido = F.unix_timestamp("inicio") - F.unix_timestamp("abertura")
ate_desistir = F.unix_timestamp("fim") - F.unix_timestamp("abertura")
atend_calc = F.unix_timestamp("fim") - F.unix_timestamp("inicio")

# Agrupa um tempo (em segundos) em faixas de LARGURA minutos, até o TETO.
def faixa_min(col):
    b = F.floor(F.col(col) / 60 / LARGURA) * LARGURA
    return F.when(F.col(col).isNotNull(), F.least(b, F.lit(TETO)))

f = (ch.join(hr, "id_chamado", "left")
       .join(pe, "id_chamado", "left")
       .join(mo, "motivo", "left")
       .withColumn("mes", F.trunc(F.to_date("abertura"), "month"))
       .withColumn("abandono", F.coalesce("abandono", F.lit(False)))
       .withColumn("qtd_respostas", F.coalesce("qtd_respostas", F.lit(0)))
       .withColumn("tem_nota", F.col("nota").isNotNull())
       .withColumn("espera_seg", F.when(humano & ~F.col("abandono"), F.greatest(F.col("tempo_espera"), espera_atendido)))
       .withColumn("espera_ate_desistir_seg", F.when(F.col("abandono"), ate_desistir))
       .withColumn("atend_seg", F.greatest(F.col("tempo_atendimento"), atend_calc))
       .withColumn("espera_corrigida", humano & ~F.col("abandono")
            & ~F.col("espera_seg").eqNullSafe(F.col("tempo_espera")))
       .withColumn("atend_corrigido", ~F.col("atend_seg").eqNullSafe(F.col("tempo_atendimento")))
       .withColumn("faixa_espera_min", faixa_min("espera_seg"))
       .withColumn("faixa_atend_min", faixa_min("atend_seg"))
       .select("id_chamado", "mes", "canal", "motivo", "categoria", "criticidade", "resolvido",
               "abandono", "nota", "tem_nota", "qtd_respostas",
               "espera_seg", "atend_seg", "espera_ate_desistir_seg",
               "faixa_espera_min", "faixa_atend_min",
               "tempo_espera", "tempo_atendimento", "espera_corrigida", "atend_corrigido"))

n_ch, n_f = ch.count(), f.count()
print("Chamados:", n_ch, "| Chamados_nota:", n_f)
assert n_f == n_ch, "o join duplicou ou perdeu chamados!"

# Idempotente (lê das silvers e sobrescreve)
(f.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable(f"{g}.chamados_nota"))

# Resumo
t = spark.table(f"{g}.chamados_nota")
print("Sem data:", t.filter("mes is null").count(),
      "| Com nota:", t.filter("tem_nota").count(),
      "| Sem categoria:", t.filter("categoria is null").count(),
      "| Abandonos:", t.filter("abandono").count(),
      "| Abandonos com nota:", t.filter("abandono and tem_nota").count())
print("Espera corrigida:", t.filter("espera_corrigida").count(),
      "| Atendimento corrigido:", t.filter("atend_corrigido").count())

display(t.orderBy("id_chamado").limit(20))

# média registrada x média corrigida (só canais humanos)
th = t.filter("canal in ('atendimento inicial', 'atendimento especializado')")
display(th.groupBy("canal").agg(
    F.round(F.avg("tempo_espera") / 60, 1).alias("espera_media_registrada_min"),
    F.round(F.avg("espera_seg") / 60, 1).alias("espera_media_corrigida_min"),
    F.round(F.avg("tempo_atendimento") / 60, 1).alias("atend_medio_registrado_min"),
    F.round(F.avg("atend_seg") / 60, 1).alias("atend_medio_corrigido_min")))

Aqui estão algumas analises que podem ser feita usando essa tabela

In [0]:
from pyspark.sql import functions as F
g = "rocket_lab.vcredit_gold"

t = spark.table(f"{g}.chamados_nota").filter("tem_nota")
t_all = spark.table(f"{g}.chamados_nota").filter("mes is not null")

# 1) nota x tempo médio, resolvido e não resolvido
display(t.groupBy("nota", "resolvido").agg(
    F.count("*").alias("chamados"),
    F.round(F.avg("espera_seg") / 60, 1).alias("espera_media_min"),
    F.round(F.avg("atend_seg") / 60, 1).alias("atendimento_medio_min"))
  .orderBy("nota", "resolvido"))

# 2) nota x faixa de espera
display(t.filter("faixa_espera_min is not null")
          .groupBy("faixa_espera_min", "nota").count().orderBy("faixa_espera_min", "nota"))

# 3) nota média por faixa de espera (onde a nota despenca)
display(t.filter("faixa_espera_min is not null").groupBy("faixa_espera_min")
          .agg(F.round(F.avg("nota"), 2).alias("nota_media"), F.count("*").alias("chamados"))
          .orderBy("faixa_espera_min"))

# 4) nota x faixa de atendimento
display(t.filter("faixa_atend_min is not null")
          .groupBy("faixa_atend_min", "nota").count().orderBy("faixa_atend_min", "nota"))

# 5) nota ao longo do tempo
display(t_all.filter("tem_nota").groupBy("mes", "nota").count().orderBy("mes", "nota"))
display(t_all.groupBy("mes").agg(
    F.count("*").alias("chamados"),
    F.round(F.avg(F.col("tem_nota").cast("int")) * 100, 1).alias("taxa_resposta_pct"),
    F.round(F.avg("nota"), 2).alias("nota_media")).orderBy("mes"))

# 6) tipo de chamado ao longo do tempo
display(t_all.groupBy("mes", "categoria").count().orderBy("mes", "categoria"))